In [ ]:
import os, sys
CARLA_ROOT = "/home/fypits25/Documents/tfddcarla/carla"
WORK_DIR = "/home/fypits25/Documents/tfddcarla/"
sys.path.extend([
    f"{CARLA_ROOT}/PythonAPI/carla",
    f"{WORK_DIR}/scenario_runner",
    f"{WORK_DIR}/leaderboard",
    f"{WORK_DIR}/team_code_transfuser"
])

In [23]:
import os
import json
import torch
from team_code_transfuser.model import LidarCenterNet
from team_code_transfuser.config import GlobalConfig
import numpy as np
from PIL import Image
from team_code_transfuser.data import lidar_to_histogram_features, draw_target_point

In [8]:
path_to_conf_file = "/home/fypits25/Documents/tfddcarla/model_ckpt/diffusiondrive"
model_file = "model_10.pth"

args_file = open(os.path.join(path_to_conf_file, 'args.txt'), 'r')
args = json.load(args_file)
args_file.close()

config = GlobalConfig(setting='eval')

In [9]:
if ('sync_batch_norm' in args):
    config.sync_batch_norm = bool(args['sync_batch_norm'])
if ('use_point_pillars' in args):
    config.use_point_pillars = args['use_point_pillars']
if ('n_layer' in args):
    config.n_layer = args['n_layer']
if ('use_target_point_image' in args):
    config.use_target_point_image = bool(args['use_target_point_image'])
if ('use_velocity' in args):
    use_velocity = bool(args['use_velocity'])
else:
    use_velocity = True

if ('image_architecture' in args):
    image_architecture = args['image_architecture']
else:
    image_architecture = 'resnet34'

if ('lidar_architecture' in args):
    lidar_architecture = args['lidar_architecture']
else:
    lidar_architecture = 'resnet18'

if ('backbone' in args):
    backbone = args['backbone']  # Options 'geometric_fusion', 'transFuser', 'late_fusion', 'latentTF'
else:
    backbone = 'transFuser'  # Options 'geometric_fusion', 'transFuser', 'late_fusion', 'latentTF'

# Load model files
    
print(os.path.join(path_to_conf_file, model_file))
net = LidarCenterNet(config, 'cuda',backbone, "diffusiondrive", image_architecture, lidar_architecture, use_velocity)
if(config.sync_batch_norm == True):
    net = torch.nn.SyncBatchNorm.convert_sync_batchnorm(net) # Model was trained with Sync. Batch Norm. Need to convert it otherwise parameters will load incorrectly.
state_dict = torch.load(os.path.join(path_to_conf_file, model_file), map_location='cuda:0')
state_dict = {k[7:]: v for k, v in state_dict.items()} # Removes the .module coming from the Distributed Training. Remove this if you want to evaluate a model trained without DDP.
net.load_state_dict(state_dict, strict=False)
net.cuda()
net.eval()



/home/fypits25/Documents/tfddcarla/model_ckpt/diffusiondrive/model_10.pth
 LidarEncoder got regnety_032
RegNet(
  (stem): ConvBnAct(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (act): ReLU(inplace=True)
    )
  )
  (s1): RegStage(
    (b1): Bottleneck(
      (conv1): ConvBnAct(
        (conv): Conv2d(32, 72, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNormAct2d(
          72, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (act): ReLU(inplace=True)
        )
      )
      (conv2): ConvBnAct(
        (conv): Conv2d(72, 72, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=3, bias=False)
        (bn): BatchNormAct2d(
          72, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (act): ReLU(inplace=True)
        )
      )
      (se): SEModule(
        (fc1): Conv

LidarCenterNet(
  (_bev_downscale): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
  (_status_encoding): Linear(in_features=6, out_features=256, bias=True)
  (_keyval_embedding): Embedding(65, 256)
  (_query_embedding): Embedding(31, 256)
  (bev_proj): Sequential(
    (0): Linear(in_features=320, out_features=256, bias=True)
    (1): ReLU(inplace=True)
    (2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (_tf_decoder): TransformerDecoder(
    (layers): ModuleList(
      (0): TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=1024, o

In [14]:
with open('/home/fypits25/Documents/tfddcarla/results_test/measurements/0000.json', 'r') as f:
    data = json.load(f)
print(data['x_command'])
print(data['y_command'])

2.0250957794838733
26.379364049626464


In [24]:
# some are not the actual input to the model in submission agent

# image
image = torch.from_numpy(np.array(Image.open("results_test/rgb/0000.png"))).unsqueeze(0).to('cuda', dtype=torch.float32) 

# lidar
lidar = np.load("/home/fypits25/Documents/tfddcarla/results_test/lidar/0000.npy", allow_pickle=True)
lidar = lidar[1][:, :3]
lidar[:, 1] *= -1  # invert
lidar = torch.from_numpy(lidar_to_histogram_features(lidar)).unsqueeze(0)
lidar_degrees = [lidar.to('cuda', dtype=torch.float32)]
lidar_bev = torch.cat(lidar_degrees[::-1], dim=1)

# target point, target point image
with open('/home/fypits25/Documents/tfddcarla/results_test/measurements/0000.json', 'r') as f:
    data = json.load(f)
target_point = [data['x_command'], data['y_command']]
target_point= [torch.FloatTensor([target_point[0]]),
                            torch.FloatTensor([target_point[1]])]
target_point = torch.stack(target_point, dim=1).to('cuda', dtype=torch.float32)

target_point_image_degrees = []
target_point_degrees = []

degree = 0
rad = np.deg2rad(degree)
degree_matrix = np.array([[np.cos(rad), np.sin(rad)],
                          [-np.sin(rad), np.cos(rad)]])

current_target_point = (degree_matrix @ target_point[0].cpu().numpy().reshape(2, 1)).T
target_point_image = draw_target_point(current_target_point[0])
target_point_image = torch.from_numpy(target_point_image)[None].to('cuda', dtype=torch.float32)
target_point_image_degrees.append(target_point_image)
target_point_degrees.append(torch.from_numpy(current_target_point))

target_point_image = torch.cat(target_point_image_degrees, dim=0)
target_point = torch.cat(target_point_degrees, dim=0).to('cuda', dtype=torch.float32)


# kinematics
ego_vel = torch.FloatTensor([data['speed']]).to('cuda', dtype=torch.float32).reshape(1,1) # used by controller
ego_acc = data['acceleration']
theta = torch.tensor(data['theta'], dtype=torch.float32)


In [27]:
print(lidar.device)
print(target_point_image.device)

cpu
cuda:0


In [28]:
# forward pass
with torch.no_grad():
    pred_wps = []
    bounding_boxes = []
    
    rotated_bb = []
    if (backbone == 'transFuser'):
        pred_wp, _ = net.forward_ego(rgb=image, lidar_bev=lidar_bev, target_point=target_point,
                target_point_image=target_point_image,
                ego_vel=ego_vel, 
                ego_acc=  torch.from_numpy(np.array(ego_acc)), 
                theta = theta, 
                save_path="/home/fypits25/Documents/tfddcarla/vis_save_path", 
                )
        
    

RuntimeError: Given groups=1, weight of size [32, 3, 3, 3], expected input[1, 480, 2880, 3] to have 3 channels, but got 480 channels instead